## Example 1: Basic HTTP client creation and configuration

In [ ]:
import httpx

# Example 1: Basic HTTP client creation and configuration
print("Example 1: HTTP client creation")

# Create basic HTTP client
client = httpx.Client()
print(f"HTTP client created: {type(client).__name__}")

# Create client with custom configuration
custom_client = httpx.Client(
    timeout=30.0,
    follow_redirects=True,
    headers={"User-Agent": "httpx-test/1.0"}
)

print(f"Custom client timeout: {custom_client.timeout}")
print(f"Follow redirects: {custom_client.follow_redirects}")
print(f"Default headers: {len(custom_client.headers)} headers")

assert custom_client.timeout == httpx.Timeout(30.0)
assert custom_client.follow_redirects is True and client.follow_redirects is False
assert custom_client.headers["user-agent"] == "httpx-test/1.0"
# The default timeout is 5 seconds for every phase
assert client.timeout == httpx.Timeout(5.0)

# Close clients
client.close()
custom_client.close()

print("HTTP client creation completed successfully")

## Example 2: Async HTTP client usage

In [ ]:
import httpx

# Example 2: Async HTTP client usage
print("Example 2: Async HTTP client")

# Create async HTTP client
async_client = httpx.AsyncClient()
print(f"Async client created: {type(async_client).__name__}")

# Create limits configuration
limits = httpx.Limits(max_keepalive_connections=5, max_connections=10)
print(f"Limits created: {type(limits).__name__}")
print(f"Max connections: {limits.max_connections}")
print(f"Max keepalive: {limits.max_keepalive_connections}")

# Create async client with configuration
async_custom = httpx.AsyncClient(
    timeout=httpx.Timeout(10.0, connect=5.0),
    limits=limits
)

print(f"Async client timeout: {async_custom.timeout}")
print(f"Client configured with custom limits")

# Demonstrate async context manager
async def demo_async_client():
    async with httpx.AsyncClient() as client:
        print(f"Context manager client: {type(client).__name__}")
        return "Async context completed"

# Run async demo
result = await demo_async_client()
print(result)

assert result == "Async context completed"
assert limits.max_connections == 10 and limits.max_keepalive_connections == 5
timeout = async_custom.timeout
assert (timeout.connect, timeout.read, timeout.write, timeout.pool) == (5.0, 10.0, 10.0, 10.0)
await async_client.aclose()
await async_custom.aclose()
assert async_client.is_closed and async_custom.is_closed

## Example 3: HTTP request and response objects

In [ ]:
import httpx

# Example 3: HTTP request and response objects
print("Example 3: Request and response objects")

# Create HTTP request
request = httpx.Request(
    method="POST",
    url="https://httpbingo.org/post",
    headers={"Content-Type": "application/json"},
    content=b'{"message": "test"}'
)

print(f"Request created: {type(request).__name__}")
print(f"Method: {request.method}")
print(f"URL: {request.url}")
print(f"Headers: {len(request.headers)} headers")
print(f"Content length: {len(request.content)}")

# Create HTTP response (mock)
response = httpx.Response(
    status_code=200,
    headers={"content-type": "application/json"},
    content=b'{"status": "success"}',
    request=request
)

print(f"Response created: {type(response).__name__}")
print(f"Status code: {response.status_code}")
print(f"Content type: {response.headers.get('content-type')}")
print(f"Response text: {response.text}")

assert request.method == "POST" and request.url.path == "/post"
# The request fills in Host and Content-Length from the URL and body
assert request.headers["host"] == "httpbingo.org" and request.headers["content-length"] == "19"
assert response.json() == {"status": "success"} and response.is_success
assert response.reason_phrase == "OK"
# raise_for_status() raises on 4xx and 5xx responses
try:
    httpx.Response(404, request=request).raise_for_status()
    raise AssertionError("expected HTTPStatusError")
except httpx.HTTPStatusError as e:
    assert e.response.status_code == 404

## Example 4: Authentication and headers

In [ ]:
import httpx

# Example 4: Authentication and headers
print("Example 4: Authentication and headers")

# Create basic authentication
auth = httpx.BasicAuth("username", "password")
print(f"Basic auth created: {type(auth).__name__}")

# Create client with authentication
client = httpx.Client(auth=auth)
print(f"Client with auth: {client.auth}")

# Create custom headers
headers = httpx.Headers({
    "User-Agent": "httpx/1.0",
    "Accept": "application/json",
    "X-Custom-Header": "test-value"
})

print(f"Headers created: {type(headers).__name__}")
print(f"Headers count: {len(headers)}")
for name, value in headers.items():
    print(f"  {name}: {value}")

# Create cookies
cookies = httpx.Cookies({"session_id": "abc123", "theme": "dark"})
print(f"Cookies created: {type(cookies).__name__}")
print(f"Cookie count: {len(cookies)}")

# BasicAuth adds an Authorization header holding base64("username:password")
authed = next(auth.auth_flow(httpx.Request("GET", "https://example.com")))
assert authed.headers["authorization"] == "Basic dXNlcm5hbWU6cGFzc3dvcmQ="
# Header lookups are case-insensitive
assert headers["user-agent"] == "httpx/1.0" and headers["X-CUSTOM-HEADER"] == "test-value"
assert len(headers) == 3
assert cookies["session_id"] == "abc123" and len(cookies) == 2

client.close()

## Example 5: URL handling and configuration

In [ ]:
import httpx

# Example 5: URL handling and configuration
print("Example 5: URL handling and configuration")

# Create and manipulate URLs
url = httpx.URL("https://api.example.com/v1/users")
print(f"URL created: {type(url).__name__}")
print(f"Scheme: {url.scheme}")
print(f"Host: {url.host}")
print(f"Path: {url.path}")

# Build URL with query parameters
query_url = url.copy_with(params={"page": 1, "limit": 10})
print(f"URL with params: {query_url}")

# Create client with custom configuration
custom_client = httpx.Client(
    base_url="https://api.example.com",
    timeout=30.0,
    follow_redirects=True
)

print(f"Client created: {type(custom_client).__name__}")
print(f"Base URL: {custom_client.base_url}")
print(f"Timeout: {custom_client.timeout}")
print(f"Follow redirects: {custom_client.follow_redirects}")

# Show URL components
print(f"URL components: scheme={url.scheme}, host={url.host}")

assert (url.scheme, url.host, url.path) == ("https", "api.example.com", "/v1/users")
assert str(query_url) == "https://api.example.com/v1/users?page=1&limit=10"
assert query_url.params["limit"] == "10"
assert custom_client.base_url == httpx.URL("https://api.example.com")
assert custom_client.build_request("GET", "/v1/users").url == url
custom_client.close()

# MockTransport answers requests in-process, so the full client path runs without a network
import json

def handler(request):
    return httpx.Response(201, json={"path": request.url.path, "body": json.loads(request.content)})

with httpx.Client(transport=httpx.MockTransport(handler), base_url="https://api.example.com") as mock_client:
    created = mock_client.post("/v1/users", json={"name": "alice"})
assert created.status_code == 201
assert created.json() == {"path": "/v1/users", "body": {"name": "alice"}}